In [131]:
import os
import re
import json
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import cv2

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from skimage.feature import hog


def load_env_file(dotenv_path: Path, override: bool = True) -> None:
    if not dotenv_path.exists():
        print("[ENV] .env not found at:", dotenv_path)
        return

    print("[ENV] loading:", dotenv_path)
    for line in dotenv_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if "=" not in s:
            continue

        k, v = s.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")

        if not override and k in os.environ:
            continue
        os.environ[k] = v


print("CWD:", Path.cwd())
load_env_file(Path.cwd() / ".env", override=True)


def _get_list_env(name: str, default: str = "") -> List[str]:
    val = os.getenv(name, default).strip()
    if not val:
        return []
    return [x.strip().lower() for x in val.split(",") if x.strip()]

def _get_int_list_env(name: str, default: str = "") -> List[int]:
    val = os.getenv(name, default).strip()
    if not val:
        return []
    return [int(x.strip()) for x in val.split(",") if x.strip()]


@dataclass
class Ex1Config:
    data_dir: Path
    ex1_dir: Path
    ann_path: Path

    seed: int = 42

    # Split control (per person)
    train_per_person: int = 150
    test_per_person: int = 50

    train_per_background: int = 15
    test_per_background: int = 5



    # Labeling / persons
    label_mode: str = "by_name"     # by_name / by_order
    category_order: List[str] = None  # used for by_order
    images_per_person: int = 0      # used if person_counts not provided
    person_counts: List[int] = None # explicit counts per person (by_order)
    background_counts: List[int] = None # explicit counts per background (by_order)

    # Classes mapping
    class0_persons: List[str] = None
    class1_persons: List[str] = None

    # Image preprocessing
    use_annotations: bool = True
    crop_margin: float = 0.10
    out_size: int = 96

    # HOG params
    hog_orientations: int = 9
    hog_pixels_per_cell: int = 8
    hog_cells_per_block: int = 2

    # Debug
    debug_max_show: int = 6


CFG = Ex1Config(
    data_dir=Path(os.getenv("DATA_DIR", "data")),
    ex1_dir=Path(os.getenv("EX1_DIR", "data/ex1")),
    ann_path=Path(os.getenv("EX1_ANN", "data/ex1/annotations.txt")),

    seed=int(os.getenv("SEED", "42")),

    train_per_person=int(os.getenv("EX1_TRAIN_PER_PERSON", "150")),
    test_per_person=int(os.getenv("EX1_TEST_PER_PERSON", "50")),

    train_per_background=int(os.getenv("EX1_TRAIN_PER_BACKGROUND", "15")),
    test_per_background=int(os.getenv("EX1_TEST_PER_BACKGROUND", "5")),

    label_mode=os.getenv("EX1_LABEL_MODE", "by_name").strip().lower(),
    category_order=_get_list_env("EX1_PERSON_ORDER"),
    images_per_person=int(os.getenv("EX1_IMAGES_PER_PERSON", "0")),
    person_counts=_get_int_list_env("EX1_PERSON_COUNTS"),
    background_counts=_get_int_list_env("EX1_BACKGROUND_COUNTS"),

    class0_persons=_get_list_env("EX1_CLASS0_PERSONS"),
    class1_persons=_get_list_env("EX1_CLASS1_PERSONS"),

    use_annotations=os.getenv("EX1_USE_ANN", "0") == "1",
    crop_margin=float(os.getenv("EX1_CROP_MARGIN", "0.10")),
    out_size=int(os.getenv("EX1_OUT_SIZE", "96")),

    hog_orientations=int(os.getenv("EX1_HOG_ORI", "9")),
    hog_pixels_per_cell=int(os.getenv("EX1_HOG_PPC", "8")),
    hog_cells_per_block=int(os.getenv("EX1_HOG_CPB", "2")),

    debug_max_show=int(os.getenv("DEBUG_MAX_SHOW", "300")),
)

random.seed(CFG.seed)
np.random.seed(CFG.seed)

print("CFG:", CFG)
print("Class0:", CFG.class0_persons)
print("Class1:", CFG.class1_persons)

CWD: c:\Users\drago\Desktop\Lucrare2
[ENV] loading: c:\Users\drago\Desktop\Lucrare2\.env
CFG: Ex1Config(data_dir=WindowsPath('data'), ex1_dir=WindowsPath('Lucrare2/data/ex1'), ann_path=WindowsPath('Lucrare2/data/ex1/annotations.txt'), seed=42, train_per_person=150, test_per_person=50, train_per_background=15, test_per_background=5, label_mode='by_order', category_order=['faces', 'backgrounds'], images_per_person=200, person_counts=[200, 20], background_counts=[], class0_persons=['1'], class1_persons=['2'], use_annotations=False, crop_margin=0.1, out_size=96, hog_orientations=9, hog_pixels_per_cell=8, hog_cells_per_block=2, debug_max_show=300)
Class0: ['1']
Class1: ['2']


In [132]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".pgm", ".ppm"}

def list_images(ex1_dir: Path) -> List[Path]:
    files = []
    for p in ex1_dir.rglob("*"):
        if "__macosx" in str(p).lower():
            continue
        if p.name.startswith("._"):
            continue
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            files.append(p)
    return files

def numeric_sort_key(p: Path):
    # "0001" -> 1
    m = re.match(r"^\D*(\d+)$", p.stem)
    return int(m.group(1)) if m else p.name.lower()




In [133]:
# hard checks for paths
if not CFG.ex1_dir.exists():
    raise FileNotFoundError(f"EX1_DIR does not exist: {CFG.ex1_dir.resolve()}")
if CFG.use_annotations and not CFG.ann_path.exists():
    raise FileNotFoundError(f"EX1_ANN does not exist: {CFG.ann_path.resolve()}")

images = list_images(CFG.ex1_dir)
images = sorted(images, key=numeric_sort_key)

anns = parse_annotations(CFG.ann_path) if CFG.use_annotations else {}

print("Found images:", len(images))
print("Found annotations:", len(anns))

def get_label(person_id: str) -> Optional[int]:
    pid = person_id.lower()
    if pid in CFG.class0_persons:
        return 0
    if pid in CFG.class1_persons:
        return 1
    return None


# counts per person
if CFG.person_counts:
    counts = CFG.person_counts
else:
    if CFG.images_per_person <= 0:
        raise ValueError("Setează EX1_IMAGES_PER_PERSON (ex: 55) sau EX1_PERSON_COUNTS.")
    counts = [CFG.images_per_person] * len(CFG.category_order)


train_paths, test_paths = [], []
train_y, test_y = [], []

start = 0
for person, cnt in zip(CFG.category_order, counts):
    seg = images[start:start+cnt]
    start += cnt

    y = get_label(person)
    

    tr = seg[:CFG.train_per_person]
    te = seg[CFG.train_per_person:CFG.train_per_person + CFG.test_per_person]


    train_paths += tr
    train_y += [y] * len(tr)
    test_paths += te
    test_y += [y] * len(te)

print("Persons kept:", len(CFG.category_order))
print("Persons:", CFG.category_order)
print("Train:", len(train_paths), "Test:", len(test_paths))


Found images: 220
Found annotations: 0
Persons kept: 2
Persons: ['faces', 'backgrounds']
Train: 170 Test: 50


In [134]:
def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def crop_with_margin(img: np.ndarray, bbox: Tuple[int,int,int,int], margin: float) -> np.ndarray:
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    mx = int(bw * margin)
    my = int(bh * margin)

    nx1 = clamp(x1 - mx, 0, w - 1)
    ny1 = clamp(y1 - my, 0, h - 1)
    nx2 = clamp(x2 + mx, 1, w)
    ny2 = clamp(y2 + my, 1, h)

    if nx2 <= nx1 or ny2 <= ny1:
        return img
    return img[ny1:ny2, nx1:nx2]

def load_face(img_path: Path) -> np.ndarray:
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot read: {img_path}")

    if CFG.use_annotations:
        key = img_path.name
        if key in anns:
            img = crop_with_margin(img, anns[key], CFG.crop_margin)
        else:
            # annotation missing -> still proceed with full image
            pass

    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = cv2.resize(img, (CFG.out_size, CFG.out_size), interpolation=cv2.INTER_AREA)
    return img

# sanity
for p in train_paths[:CFG.debug_max_show]:
    face = load_face(p)
    print(p.name, face.shape, face.min(), face.max())


001.jpg (96, 96) 31 234
002.jpg (96, 96) 22 239
003.jpg (96, 96) 29 221
004.jpg (96, 96) 18 226
005.jpg (96, 96) 26 226
006.jpg (96, 96) 6 220
007.jpg (96, 96) 13 223
008.jpg (96, 96) 18 205
009.jpg (96, 96) 8 247
010.jpg (96, 96) 34 218
011.jpg (96, 96) 25 212
012.jpg (96, 96) 14 228
013.jpg (96, 96) 25 242
014.jpg (96, 96) 26 223
015.jpg (96, 96) 59 244
016.jpg (96, 96) 8 224
017.jpg (96, 96) 13 244
018.jpg (96, 96) 23 215
019.jpg (96, 96) 30 222
020.jpg (96, 96) 34 237
021.jpg (96, 96) 16 216
022.jpg (96, 96) 26 243
023.jpg (96, 96) 22 224
024.jpg (96, 96) 15 234
025.jpg (96, 96) 16 226
026.jpg (96, 96) 44 240
027.jpg (96, 96) 16 226
028.jpg (96, 96) 28 226
029.jpg (96, 96) 16 232
030.jpg (96, 96) 29 226
031.jpg (96, 96) 7 231
032.jpg (96, 96) 37 233
033.jpg (96, 96) 20 231
034.jpg (96, 96) 6 237
035.jpg (96, 96) 42 221
036.jpg (96, 96) 17 206
037.jpg (96, 96) 12 247
038.jpg (96, 96) 6 225
039.jpg (96, 96) 17 226
040.jpg (96, 96) 9 215
041.jpg (96, 96) 19 233
042.jpg (96, 96) 8 245


In [ ]:
def hog_features(gray_img: np.ndarray) -> np.ndarray:
    feat = hog(
        gray_img,
        orientations=CFG.hog_orientations,
        pixels_per_cell=(CFG.hog_pixels_per_cell, CFG.hog_pixels_per_cell),
        cells_per_block=(CFG.hog_cells_per_block, CFG.hog_cells_per_block),
        block_norm="L2-Hys",
        feature_vector=True
    )
    return feat.astype(np.float32)

def build_X(paths: List[Path]) -> np.ndarray:
    if len(paths) == 0:
        raise ValueError("build_X got 0 paths (train/test empty). Check label/split config.")
    feats = []
    for p in paths:
        img = load_face(p)
        feats.append(hog_features(img))
    return np.vstack(feats)

X_train = build_X(train_paths)
X_test  = build_X(test_paths)

# Validate labels: ensure no None values (unlabeled persons)
if any(v is None for v in (train_y + test_y)):
    raise ValueError("Found unlabeled entries in train_y/test_y.",
Set EX1_CLASS0_PERSONS/EX1_CLASS1_PERSONS or EX1_PERSON_ORDER to include all persons.")

y_train = np.array(train_y, dtype=np.int32)
y_test  = np.array(test_y, dtype=np.int32)

print("X_train:", X_train.shape, "X_test:", X_test.shape)


TypeError: int() argument must be a string, a bytes-like object or a number, not 'NoneType'

In [136]:
model = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", LinearSVC(C=1.0))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)

print("Accuracy:", acc)
print("Confusion:\n", cm)
print(classification_report(y_test, pred, digits=4))


ValueError: Found input variables with inconsistent numbers of samples: [170, 80]

In [ ]:
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

rows = []
for p, y_true, y_hat in zip(test_paths, y_test.tolist(), pred.tolist()):
    rows.append({"file": p.name, "y_true": int(y_true), "y_pred": int(y_hat)})

with (out_dir / "ex1_preds.json").open("w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print("Saved:", out_dir / "ex1_preds.json")


Saved: outputs\ex1_preds.json


In [ ]:
# Export processed images into outputs/e001.jpg..e220.jpg
# Mapping: training -> e001-e150, testing -> e151-e200, training -> e201-e215, testing -> e216-e220
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# counters for source lists (train_paths and test_paths were built earlier)
_train_i = 0
_test_i = 0
mappings = [("train", 1, 150), ("test", 151, 200), ("train", 201, 215), ("test", 216, 220)]
rows = []
for src_type, start, end in mappings:
    for num in range(start, end + 1):
        if src_type == "train":
            if _train_i >= len(train_paths):
                raise IndexError(f"Not enough train images: need at least {_train_i+1}, have {len(train_paths)}")
            src = train_paths[_train_i]
            _train_i += 1
        else:
            if _test_i >= len(test_paths):
                raise IndexError(f"Not enough test images: need at least {_test_i+1}, have {len(test_paths)}")
            src = test_paths[_test_i]
            _test_i += 1

        # load_face returns a grayscale, resized image (CFG.out_size x CFG.out_size)
        face = load_face(src)
        fname = f"e{num:03d}.jpg"
        cv2.imwrite(str(out_dir / fname), face)
        rows.append({"out": fname, "src": src.name, "type": src_type})

# Save mapping for reference
with (out_dir / "e_mapping.json").open("w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print("Export complete. Written:", len(rows), "images to", out_dir)
print("Train consumed:", _train_i, "Test consumed:", _test_i)
